# 🧠 강화학습 5강 — 두뇌의 구조: DQN
### 카트폴 · 벽돌깨기 · 루나랜더 착륙, 그리고 허깅페이스에 내 모델 올리기

**AI CITY BUILDERS · Connect AI LAB**

4강까지 두뇌는 **표(Q표)** 였습니다. 얼음 호수는 64칸 × 행동 4개 = 256칸이면 충분했죠.
그런데 상태가 **연속된 숫자**이거나 **화면 전체**라면 칸을 셀 수 없습니다. 그래서 표 대신 **신경망**이 Q 를 계산합니다 — **DQN (Deep Q-Network)**.

오늘은 **DQN 학습 함수 하나**를 만들고, **환경만 바꿔서** 세 게임을 풉니다.

| 게임 | 상태 | 행동 | 보상 |
|---|---|---|---|
| 🛒 카트폴 | 숫자 4개 (위치 · 속도 · 각도 · 각속도) | 2개 (왼쪽 · 오른쪽) | 버틴 걸음마다 +1 |
| 🧱 벽돌깨기 | 숫자 8개 (판 · 공 위치와 속도 · 남은 벽돌 · 목숨) | 3개 (가만히 · 왼쪽 · 오른쪽) | 벽돌 +1 · 공 놓치면 −1 |
| 🌙 루나랜더 (Gymnasium) | 숫자 8개 (위치 · 속도 · 기울기 · 회전 속도 · 두 다리) | 4개 (쉬기 · 왼쪽 · 가운데 · 오른쪽 엔진) | 착지 +100~140 · 추락 −100 · 연료 − |

그리고 마지막에 내가 학습시킨 두뇌를 **허깅페이스(Hugging Face)** 에 올려, 모델 카드가 있는 **나의 첫 공개 모델**을 만듭니다.

> ▶ **런타임 → 모두 실행** (GPU 필요 없음 · 약 15분). 허깅페이스 업로드(11~12번 셀)는 토큰을 넣어야 진행됩니다.

## 🔍 DQN 이 뭐예요?

**DQN = Deep Q-Network = 「Q 를 계산하는 신경망」**

| 글자 | 뜻 |
|---|---|
| **D** · Deep | 숫자를 받아 여러 층을 거쳐 답을 계산하는 깊은 신경망 |
| **Q** · 행동가치 | 2~4강의 그 Q — 「이 상태에서 이 행동을 하면 앞으로 받을 점수의 합」을 미리 짐작한 값. 가장 큰 Q 의 행동을 고른다 |
| **N** · Network | 신경망 |

질문은 4강까지와 똑같습니다 — **「지금 이 상태에서 왼쪽은 몇 점? 오른쪽은 몇 점?」**
달라진 건 대답하는 방법뿐입니다. Q표는 **칸을 찾아서** 대답하고, DQN 은 **숫자를 넣어 계산해서** 대답합니다. 표를 채우던 일을 신경망이 배우는 거예요.

**DQN 이 배우는 한 바퀴**
1. **해 보기** — 탐험(ε)하며 행동한다 (2강)
2. **기록** — (상태 · 행동 · 보상 · 다음 상태)를 기억 상자에 쌓는다 (3강)
3. **정답 만들기** — 정답 = 보상 + γ × 다음 상태의 가장 큰 Q (4강) · 목표 두뇌가 계산
4. **차이 줄이기** — 두뇌의 예측을 정답 쪽으로 조금 당긴다 → 다시 1번으로

> 셀마다 움직이며 이해하는 해설 페이지: [aicitybuilders.com/brain](https://www.aicitybuilders.com/brain)

## CELL 01 · 준비

In [ ]:
!pip install -q "gymnasium[classic-control]>=1.0" imageio-ffmpeg huggingface_hub
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import os, math, time, json, random, base64, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym, imageio
import matplotlib, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display

글꼴경로 = next((p for p in ['/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf',
                            '/System/Library/Fonts/Supplemental/AppleGothic.ttf'] if os.path.exists(p)), None)
if 글꼴경로:
    matplotlib.font_manager.fontManager.addfont(글꼴경로)
    plt.rcParams['font.family'] = matplotlib.font_manager.FontProperties(fname=글꼴경로).get_name()
plt.rcParams['axes.unicode_minus'] = False
def 글꼴(n):
    try: return ImageFont.truetype(글꼴경로, n)
    except Exception: return ImageFont.load_default()

torch.set_num_threads(1)   # 작은 두뇌는 CPU 한 줄이 가장 빠르다
def 영상보기(화면들, 파일, fps=30, 너비=480):
    imageio.mimsave(파일, 화면들, fps=fps, macro_block_size=1)
    display(HTML(f'<video width="{너비}" controls autoplay loop muted><source src="data:video/mp4;base64,'
                 f'{base64.b64encode(open(파일, "rb").read()).decode()}" type="video/mp4"></video>'))
print('✅ 준비 완료 · torch', torch.__version__, '· gymnasium', gym.__version__)

## CELL 02 · 왜 표로는 안 될까

4강 얼음 호수의 Q표는 **64칸 × 4행동 = 256개의 숫자**였습니다.
카트폴의 상태는 숫자 4개인데, 이 숫자들은 **연속값**입니다. 소수점 둘째 자리까지만 나눠도 칸이 몇 개가 될까요?

In [ ]:
env = gym.make('CartPole-v1')
s, _ = env.reset(seed=0)
print('카트폴 상태 :', np.round(s, 4), ' ← 수레 위치 · 수레 속도 · 막대 각도 · 막대 각속도')
print('행동        :', env.action_space.n, '개 (0 왼쪽으로 밀기 · 1 오른쪽으로 밀기)')

칸 = 1
for 이름, 범위 in [('수레 위치', 4.8), ('수레 속도', 6.0), ('막대 각도', 0.42), ('막대 각속도', 6.0)]:
    n = int(범위 / 0.01); 칸 *= n
    print(f'  {이름:<6} 0.01 간격으로 나누면 {n:>4}칸')
print(f'\n표로 만들면 상태 칸 수 ≈ {칸:,}개 × 행동 2개')
print('얼음 호수 256개와 비교하면', f'{칸*2/256:,.0f}', '배 — 평생 걸어도 다 못 채웁니다.')
print('\n👉 그래서 표 대신 「숫자를 넣으면 Q값이 나오는 함수」, 신경망을 씁니다.')

## CELL 03 · 두뇌 = 신경망

표는 「칸을 찾아서 꺼내는 것」이고, 신경망은 「숫자를 넣으면 계산해서 내놓는 것」입니다.

```
상태 숫자 4개 ──▶ [ 128 ] ──▶ [ 128 ] ──▶ Q(왼쪽) · Q(오른쪽)
```

처음 본 상태라도 **비슷한 상태에서 배운 것**을 바탕으로 Q 를 계산할 수 있습니다. 이게 표와의 가장 큰 차이입니다.

In [ ]:
def 두뇌만들기(상태수, 행동수, 은닉=128):
    return nn.Sequential(nn.Linear(상태수, 은닉), nn.ReLU(), nn.Linear(은닉, 은닉), nn.ReLU(), nn.Linear(은닉, 행동수))

뇌 = 두뇌만들기(4, 2)
print(뇌)
print('\n두뇌 안의 숫자(가중치) 개수 :', sum(p.numel() for p in 뇌.parameters()), '개')
with torch.no_grad(): q = 뇌(torch.tensor(s))
print('아직 아무것도 모르는 두뇌의 Q값 →', f'왼쪽 {q[0]:+.3f} · 오른쪽 {q[1]:+.3f}', ' (아무 의미 없는 숫자)')

## CELL 04 · DQN 학습 함수 하나 — 지금까지 배운 것 전부

| 어디서 배웠나 | DQN 안에서 | 코드 |
|---|---|---|
| 2강 탐험 | ε 를 1 → 0.02 로 줄이며 아무거나 해 보기 | `eps` |
| 3강 기억 상자 | 겪은 (상태 · 행동 · 보상 · 다음 상태)를 쌓아 두고 **무작위로 꺼내 복습** | `S, A, R, S2, D` |
| 4강 γ | 다음 상태의 가장 큰 Q × γ 를 정답에 더한다 | `감마` |
| **5강 새로 나온 것** | **목표 두뇌** — 정답을 만드는 두뇌는 가끔만 복사해서 흔들리지 않게 | `목표` |

**이 함수 하나로 세 게임을 다 풉니다.** 바뀌는 건 환경과 숫자 몇 개뿐입니다.

In [ ]:
def DQN학습(환경, 상태수, 행동수, 총걸음, 은닉=128, 학습률=1e-3, 감마=0.99, 탐험기간=0.3, 목표주기=500,
           학습시작=1000, 배치=64, 기억크기=50_000, 씨앗=0, 보고=5000, 평가=None):
    random.seed(씨앗); np.random.seed(씨앗); torch.manual_seed(씨앗)
    q = 두뇌만들기(상태수, 행동수, 은닉); 목표 = 두뇌만들기(상태수, 행동수, 은닉); 목표.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=학습률)
    S = np.zeros((기억크기, 상태수), np.float32); A = np.zeros(기억크기, np.int64); R = np.zeros(기억크기, np.float32)
    S2 = np.zeros((기억크기, 상태수), np.float32); D = np.zeros(기억크기, np.float32); k = 0; 찬 = 0
    s = 환경.reset(); 판점수 = []; 지금점수 = 0; 최고 = (-1e9, None); 평가기록 = []; t0 = time.time()
    for 걸음 in range(1, 총걸음 + 1):
        eps = max(0.02, 1 - 걸음 / (총걸음 * 탐험기간))                          # ① 탐험 (2강)
        if random.random() < eps: a = random.randrange(행동수)
        else:
            with torch.no_grad(): a = int(q(torch.from_numpy(s)).argmax())
        s2, r, 끝, 진짜끝 = 환경.step(a)
        S[k], A[k], R[k], S2[k], D[k] = s, a, r, s2, float(진짜끝); k = (k + 1) % 기억크기; 찬 = min(찬 + 1, 기억크기)   # ② 기억 상자 (3강)
        지금점수 += r; s = s2
        if 끝: 판점수.append(지금점수); 지금점수 = 0; s = 환경.reset()
        if 걸음 > 학습시작:                                                      # ③ 꺼내서 복습
            i = np.random.randint(0, 찬, 배치)
            with torch.no_grad():
                정답 = torch.from_numpy(R[i]) + 감마 * 목표(torch.from_numpy(S2[i])).max(1).values * (1 - torch.from_numpy(D[i]))   # ④ γ (4강)
            예측 = q(torch.from_numpy(S[i])).gather(1, torch.from_numpy(A[i]).unsqueeze(1)).squeeze(1)
            loss = F.smooth_l1_loss(예측, 정답); opt.zero_grad(); loss.backward(); opt.step()
        if 걸음 % 목표주기 == 0: 목표.load_state_dict(q.state_dict())               # ⑤ 목표 두뇌 (5강)
        if 걸음 % 보고 == 0:
            최근 = np.mean(판점수[-20:]) if 판점수 else 0
            줄 = f'[{걸음:>7,}걸음 | {time.time()-t0:4.0f}초 | ε {eps:.2f}] 최근 20판 평균 {최근:7.1f}'
            if 평가:
                점수 = 평가(q); 평가기록.append((걸음, 점수)); 줄 += f' | 시험 {점수:7.1f}'
                if 점수 >= 최고[0]: 최고 = (점수, {k_: v.clone() for k_, v in q.state_dict().items()}); 줄 += '  🏅'
            print(줄)
    if 평가 and 최고[1] is not None: q.load_state_dict(최고[1])                   # 시험 점수가 가장 좋았던 두뇌를 남긴다
    return q, 판점수, 평가기록

def 곡선(판점수, 제목, 기준=None):
    plt.figure(figsize=(9, 3.2)); plt.plot(판점수, alpha=.25, label='한 판 점수')
    if len(판점수) >= 20: plt.plot(range(19, len(판점수)), np.convolve(판점수, np.ones(20) / 20, 'valid'), lw=2.5, label='최근 20판 평균')
    if 기준: plt.axhline(기준, ls='--', c='gray', label=f'목표 {기준}')
    plt.xlabel('판'); plt.ylabel('점수'); plt.title(제목); plt.legend(); plt.tight_layout(); plt.show()
print('✅ DQN학습() 준비 — 이 함수 하나로 세 게임을 풉니다')

## CELL 05 · 🛒 카트폴 — 막대를 세워라

수레를 왼쪽 · 오른쪽으로 밀어 막대가 쓰러지지 않게 버팁니다. 한 걸음 버틸 때마다 +1, **최대 500점**.
1강에서 무작위로 움직였을 때는 20점 남짓이었죠.

In [ ]:
class 카트폴환경:
    def __init__(self, 씨앗=0): self.env = gym.make('CartPole-v1'); self.씨앗 = 씨앗
    def reset(self):
        self.씨앗 += 1; s, _ = self.env.reset(seed=self.씨앗); return s.astype(np.float32)
    def step(self, a):
        s, r, 쓰러짐, 시간끝, _ = self.env.step(a)
        return s.astype(np.float32), r, 쓰러짐 or 시간끝, 쓰러짐     # 500걸음을 채워 끝난 건 「진짜 끝」이 아니다

def 카트폴시험(q, 판=5):
    env = gym.make('CartPole-v1'); 합 = 0
    for 씨 in range(1000, 1000 + 판):
        s, _ = env.reset(seed=씨); 끝 = False
        while not 끝:
            with torch.no_grad(): a = int(q(torch.from_numpy(s.astype(np.float32))).argmax())
            s, r, t1, t2, _ = env.step(a); 합 += r; 끝 = t1 or t2
    return 합 / 판

print('🎲 무작위 두뇌 시험 :', 카트폴시험(두뇌만들기(4, 2)), '점')
카트폴두뇌, 카트폴점수, _ = DQN학습(카트폴환경(), 4, 2, 총걸음=50_000, 학습률=1e-3, 감마=0.99, 탐험기간=0.2, 목표주기=200, 배치=128,
                               보고=5000, 평가=카트폴시험)
카트폴최종 = 카트폴시험(카트폴두뇌, 10)
print(f'\n✅ 다 배운 카트폴 두뇌 — 처음 보는 10판 평균 {카트폴최종:.0f}점 (최대 500)')

In [ ]:
곡선(카트폴점수, '카트폴 — DQN 이 스스로 배운 기록', 500)

def 카트폴영상(q, 씨앗=7, 파일='cartpole.mp4'):
    env = gym.make('CartPole-v1', render_mode='rgb_array'); s, _ = env.reset(seed=씨앗); 화면들 = []; 점수 = 0; 끝 = False
    while not 끝:
        with torch.no_grad(): Q = q(torch.from_numpy(s.astype(np.float32))).tolist()
        a = int(np.argmax(Q)); s, r, t1, t2, _ = env.step(a); 점수 += r; 끝 = t1 or t2
        if int(점수) % 2 == 0:
            im = Image.fromarray(env.render()); d = ImageDraw.Draw(im)
            d.text((12, 10), f'걸음 {int(점수)}', fill=(20, 20, 20), font=글꼴(20))
            for j, 이름 in enumerate(['왼쪽', '오른쪽']):
                d.text((12, 40 + j * 24), f'{"▶" if j == a else "  "} Q({이름}) {Q[j]:6.2f}', fill=(37, 99, 235) if j == a else (120, 120, 120), font=글꼴(16))
            화면들.append(np.array(im))
    영상보기(화면들, 파일, fps=30); return 점수

print('🎬 다 배운 두뇌가 버틴 걸음 :', 카트폴영상(카트폴두뇌))

## CELL 06 · 🧱 벽돌깨기 — 같은 함수, 다른 게임

[aicitybuilders.com/dqn](https://www.aicitybuilders.com/dqn) 과 **한 글자도 다르지 않은** 미니 벽돌깨기입니다. 상태는 화면이 아니라 **숫자 8개**(요약본).
카트폴 때와 **똑같은 `DQN학습()`** 을 부릅니다. 바뀐 건 환경 클래스와 숫자 몇 개뿐.

In [ ]:
# 미니 벽돌깨기 — 사이트(aicitybuilders.com/dqn)와 같은 규칙
W, H = 300, 400
COLS, ROWS, BW, BH, TOP = 10, 6, 30, 12, 60
PW, PY, PSPEED = 50, 370, 6
SPEED, LIVES, MAX_STEPS = 4, 3, 3000
LAUNCH = [-2, -1, 0.5, 1.5, 2.5, -2.5, 1, -0.5]
벽돌행동 = ['가만히', '왼쪽', '오른쪽']

class Game:
    def __init__(self, seed=0):
        self.seed = seed; self.px = W / 2; self.bx = self.by = self.vx = self.vy = 0.0
        self.bricks = [1] * (COLS * ROWS); self.left = COLS * ROWS
        self.lives = LIVES; self.score = 0; self.steps = 0; self.launches = 0; self.done = False; self._launch()
    def _launch(self):
        vx = LAUNCH[(self.seed * 3 + self.launches) % len(LAUNCH)]; self.launches += 1
        self.bx = self.px; self.by = PY - 10; self.vx = vx; self.vy = -math.sqrt(SPEED * SPEED - vx * vx)
    def state(self):
        return [self.px / W, self.bx / W, self.by / H, self.vx / SPEED, self.vy / SPEED,
                (self.bx - self.px) / W, self.left / (COLS * ROWS), self.lives / LIVES]
    def step(self, action):
        if self.done: return 0
        reward = 0
        if action == 1: self.px -= PSPEED
        elif action == 2: self.px += PSPEED
        self.px = min(max(self.px, PW / 2), W - PW / 2)
        self.bx += self.vx; self.by += self.vy
        if self.bx < 0: self.bx = -self.bx; self.vx = -self.vx
        if self.bx > W: self.bx = 2 * W - self.bx; self.vx = -self.vx
        if self.by < 0: self.by = -self.by; self.vy = -self.vy
        if TOP <= self.by < TOP + ROWS * BH:
            c = math.floor(self.bx / BW); r = math.floor((self.by - TOP) / BH); i = r * COLS + c
            if 0 <= c < COLS and self.bricks[i] == 1:
                self.bricks[i] = 0; self.left -= 1; self.score += 1; reward += 1; self.vy = -self.vy
        if self.vy > 0 and PY <= self.by <= PY + 8 and self.px - PW / 2 - 4 <= self.bx <= self.px + PW / 2 + 4:
            off = min(max((self.bx - self.px) / (PW / 2), -1), 1)
            self.vx = off * 3; self.vy = -math.sqrt(SPEED * SPEED - self.vx * self.vx); self.by = PY
        if self.by > H:
            self.lives -= 1; reward -= 1
            if self.lives <= 0: self.done = True
            else: self._launch()
        if self.left == 0: self.done = True
        self.steps += 1
        if self.steps >= MAX_STEPS: self.done = True
        return reward

class 벽돌환경:
    def reset(self):
        self.g = Game(random.randrange(1000)); return np.array(self.g.state(), np.float32)
    def step(self, a):
        r = self.g.step(a); return np.array(self.g.state(), np.float32), r, self.g.done, self.g.done or r < 0

def 벽돌시험(q, 씨앗들=(0, 1, 2, 3, 4)):      # 사이트 대회와 같은 씨앗 5개
    합 = 0
    for 씨 in 씨앗들:
        g = Game(씨)
        while not g.done:
            with torch.no_grad(): g.step(int(q(torch.tensor(g.state())).argmax()))
        합 += g.score
    return 합 / len(씨앗들)

print('🎲 무작위 두뇌 시험 :', 벽돌시험(두뇌만들기(8, 3, 64)), '개')
벽돌두뇌, 벽돌점수, _ = DQN학습(벽돌환경(), 8, 3, 총걸음=150_000, 은닉=64, 학습률=5e-4, 감마=0.99, 탐험기간=0.4,
                            목표주기=1000, 학습시작=2000, 보고=15_000, 평가=벽돌시험)
벽돌최종 = 벽돌시험(벽돌두뇌)
print(f'\n✅ 다 배운 벽돌깨기 두뇌 — 대회 씨앗 5판 평균 벽돌 {벽돌최종:.1f}개 (전체 60개)')

In [ ]:
곡선(벽돌점수, '벽돌깨기 — 같은 DQN, 다른 게임')
색 = [(248, 113, 113), (251, 146, 60), (250, 204, 21), (74, 222, 128), (56, 189, 248), (167, 139, 250)]
def 벽돌그리기(g, Q=None, a=None):
    im = Image.new('RGB', (W + 170, H), (14, 18, 28)); d = ImageDraw.Draw(im)
    for i, b in enumerate(g.bricks):
        if b: r, c = divmod(i, COLS); d.rectangle([c*BW+1, TOP+r*BH+1, c*BW+BW-2, TOP+r*BH+BH-2], fill=색[r])
    d.rectangle([g.px-PW/2, PY, g.px+PW/2, PY+6], fill=(226, 232, 240)); d.ellipse([g.bx-4, g.by-4, g.bx+4, g.by+4], fill=(255, 255, 255))
    d.line([W, 0, W, H], fill=(51, 65, 85))
    d.text((W+12, 12), f'벽돌 {g.score}', fill=(255, 214, 102), font=글꼴(20)); d.text((W+12, 40), f'목숨 {g.lives} · {g.steps}걸음', fill=(160, 170, 190), font=글꼴(13))
    if Q is not None:
        lo, hi = min(min(Q), 0), max(max(Q), 0.01)
        for k in range(3):
            y = 90 + k * 50; L = int(140 * (Q[k] - lo) / (hi - lo + 1e-9))
            d.text((W+12, y), f'{벽돌행동[k]} {Q[k]:.2f}', fill=(230, 235, 245) if k == a else (140, 150, 170), font=글꼴(14))
            d.rectangle([W+12, y+20, W+12+max(L, 2), y+34], fill=(56, 189, 248) if k == a else (71, 85, 105))
    return np.array(im)

def 벽돌영상(q, 씨앗=1, 최대=1500, 파일='breakout.mp4'):
    g = Game(씨앗); 화면들 = []
    while not g.done and g.steps < 최대:
        with torch.no_grad(): Q = q(torch.tensor(g.state())).tolist()
        a = int(np.argmax(Q)); g.step(a)
        if g.steps % 2 == 0: 화면들.append(벽돌그리기(g, Q, a))
    영상보기(화면들, 파일, fps=60, 너비=470); return g.score
print('🎬 다 배운 두뇌가 깬 벽돌 :', 벽돌영상(벽돌두뇌))

## CELL 07 · (선택) 원작 아타리 벽돌깨기 — 화면을 보는 두뇌

미니 벽돌깨기는 숫자 8개를 봤습니다. 원작 아타리는 **화면 그 자체**(210 × 160 × 3 = 100,800개 숫자)를 봅니다.
그래서 두뇌 앞에 **눈(합성곱 신경망, CNN)** 이 붙습니다. 2015년 딥마인드 DQN 논문 구조 그대로입니다.

```
흑백 84×84 화면 4장 ──▶ [눈: 합성곱 3층] ──▶ [512] ──▶ Q(가만히) Q(발사) Q(오른쪽) Q(왼쪽)
```

수백만 걸음이 필요해서 코랩에서 처음부터 학습하기는 어렵습니다. **Connect AI LAB 이 미리 오래 학습시킨 두뇌**를 불러와 보기만 합니다. (설치 포함 약 1~2분 · 건너뛰어도 됩니다)

In [ ]:
!pip install -q "gymnasium[atari]>=1.0" "ale-py>=0.9"
import ale_py; gym.register_envs(ale_py)

class 발사도우미(gym.Wrapper):
    def reset(self, **kw):
        o, i = self.env.reset(**kw); o, _, _, _, i = self.env.step(1); self.목숨 = i['lives']; return o, i
    def step(self, a):
        o, r, t1, t2, i = self.env.step(a)
        if i['lives'] < self.목숨 and not (t1 or t2): o, _, t1, t2, i = self.env.step(1)
        self.목숨 = i['lives']; return o, r, t1, t2, i

class Q두뇌(nn.Module):
    def __init__(self, n=4):
        super().__init__()
        self.눈 = nn.Sequential(nn.Conv2d(4, 32, 8, 4), nn.ReLU(), nn.Conv2d(32, 64, 4, 2), nn.ReLU(), nn.Conv2d(64, 64, 3, 1), nn.ReLU(), nn.Flatten())
        self.판단 = nn.Sequential(nn.Linear(3136, 512), nn.ReLU(), nn.Linear(512, n))
    def forward(self, x): return self.판단(self.눈(x.float() / 255.0))

!wget -q -O 아타리_고수_두뇌.pt https://github.com/wonseokjung/aicb-colab/raw/main/models/breakout_dqn.pt
고수 = Q두뇌(); 고수.load_state_dict(torch.load('아타리_고수_두뇌.pt', map_location='cpu')); 고수.eval()
print('두뇌 안의 숫자 :', f'{sum(p.numel() for p in 고수.parameters()):,}', '개 (카트폴 두뇌의 약', sum(p.numel() for p in 고수.parameters()) // sum(p.numel() for p in 카트폴두뇌.parameters()), '배)')

env = gym.make('ALE/Breakout-v5', render_mode='rgb_array', frameskip=1, repeat_action_probability=0.0)
env = gym.wrappers.AtariPreprocessing(env, frame_skip=4, screen_size=84, grayscale_obs=True, noop_max=0)
env = 발사도우미(gym.wrappers.FrameStackObservation(env, 4))
o, _ = env.reset(seed=3); 화면들 = []; 점수 = 0
for t in range(2500):
    with torch.no_grad(): a = int(고수(torch.tensor(np.array(o))[None]).argmax()) if random.random() > 0.01 else random.randrange(4)
    o, r, t1, t2, _ = env.step(a); 점수 += r
    if t % 2 == 0: 화면들.append(env.unwrapped.render())
    if t1 or t2: break
영상보기(화면들, 'atari.mp4', fps=30, 너비=320)
print('🧱 원작 아타리 — 미리 학습한 두뇌 점수 :', 점수)

## CELL 08 · 🌙 진짜 루나랜더 — Gymnasium 에서 받아 오기

이번엔 우리가 만든 게임이 아니라 **Gymnasium 의 `LunarLander-v3`** 입니다. 물리 엔진(Box2D)이 들어간 진짜 시뮬레이터예요. (Gymnasium 은 OpenAI 가 만든 Gym 을 이어받은 환경 모음입니다.)
착륙선을 추진기 세 개로 조종해 깃발 사이 착지대에 사뿐히 내려야 합니다. 로켓 · 드론 · 날아오르는 자동차 — **날아오르는 것보다 내려앉는 게 어렵습니다.**

| 재료 | 루나랜더에서는 |
|---|---|
| 상태 (8개) | 위치 x · y, 속도 x · y, 기울기, 회전 속도, 왼다리 닿음, 오른다리 닿음 |
| 행동 (4개) | 0 쉬기 · 1 왼쪽 엔진 · 2 가운데 엔진 · 3 오른쪽 엔진 |
| 보상 | 착지대로 내려와 멈추면 약 +100~140 · 추락 −100 · 다리가 땅에 닿을 때마다 +10 · 가운데 엔진 한 번 −0.3 · 옆 엔진 −0.03 |
| 성공 | 한 판 **200점 이상** |

In [ ]:
!pip install -q swig
!pip install -q "gymnasium[box2d]>=1.0"
env = gym.make('LunarLander-v3', render_mode='rgb_array')
s, _ = env.reset(seed=0)
print('상태 8개 :', np.round(s, 3))
print('행동     :', env.action_space.n, '개 — 0 쉬기 · 1 왼쪽 엔진 · 2 가운데 엔진 · 3 오른쪽 엔진')
점수 = []
for 씨 in range(20):
    s, _ = env.reset(seed=씨); 끝 = False; R = 0
    while not 끝: s, r, t1, t2, _ = env.step(env.action_space.sample()); R += r; 끝 = t1 or t2
    점수.append(round(R))
print('🎲 무작위로 20판 →', 점수, '· 평균', round(np.mean(점수), 1), '(200 이상이 성공)')
s, _ = env.reset(seed=3); 화면들 = []; 끝 = False
while not 끝: s, r, t1, t2, _ = env.step(env.action_space.sample()); 끝 = t1 or t2; 화면들.append(env.render())
영상보기(화면들[::2], 'lunar_random.mp4', fps=25, 너비=420)

## CELL 09 · 같은 DQN 함수로 착륙

카트폴 · 벽돌깨기에 쓴 **`DQN학습()` 그대로**입니다. 상태 8개 · 행동 4개만 알려 줍니다. 노트북 CPU 로 약 1~4분.

In [ ]:
class 착륙환경:
    def __init__(self, 보상가공=None): self.env = gym.make('LunarLander-v3'); self.씨 = 0; self.가공 = 보상가공
    def reset(self): self.씨 += 1; s, _ = self.env.reset(seed=self.씨); self.이전 = s; return s.astype(np.float32)
    def step(self, a):
        s, r, 끝1, 끝2, _ = self.env.step(a)
        if self.가공: r = self.가공(self.이전, a, r, s, 끝1)
        self.이전 = s
        return s.astype(np.float32), r, 끝1 or 끝2, 끝1

def 착륙시험(q, 씨앗들=range(2000, 2005)):          # 채점은 언제나 「원래 보상」으로
    env = gym.make('LunarLander-v3'); 합 = 0
    for 씨 in 씨앗들:
        s, _ = env.reset(seed=씨); 끝 = False
        while not 끝:
            with torch.no_grad(): a = int(q(torch.from_numpy(s.astype(np.float32))).argmax())
            s, r, t1, t2, _ = env.step(a); 합 += r; 끝 = t1 or t2
    return 합 / len(씨앗들)

착륙두뇌, 착륙점수, _ = DQN학습(착륙환경(), 8, 4, 총걸음=200_000, 학습률=5e-4, 감마=0.99, 탐험기간=0.3, 목표주기=500, 배치=128,
                             기억크기=100_000, 보고=20_000, 평가=착륙시험)
착륙최종 = 착륙시험(착륙두뇌, range(2000, 2010))
print(f'\n✅ 다 배운 착륙 두뇌 — 처음 보는 10판 평균 {착륙최종:.1f}점 (200 이상 = 성공)')
곡선(착륙점수, '루나랜더 — 같은 DQN, 진짜 시뮬레이터', 200)
env = gym.make('LunarLander-v3', render_mode='rgb_array'); s, _ = env.reset(seed=2003); 화면들 = []; 끝 = False
while not 끝:
    with torch.no_grad(): a = int(착륙두뇌(torch.from_numpy(s.astype(np.float32))).argmax())
    s, r, t1, t2, _ = env.step(a); 끝 = t1 or t2; 화면들.append(env.render())
영상보기(화면들[::2], 'lunar.mp4', fps=25, 너비=420)

## CELL 10 · 🏆 도전 — 착륙 점수를 올려라 (강화학습 컵 #4 준비)

아래 숫자를 바꿔 다시 학습시켜 보세요. **채점은 언제나 Gymnasium 원래 보상**으로, 정해진 착륙장 5곳에서 합니다. 학습할 때 쓰는 보상은 `보상가공` 으로 바꿔도 됩니다 — 4강에서 배운 「두뇌가 믿는 보상」이죠.

| 바꿀 것 | 지금 | 생각해 볼 질문 |
|---|---|---|
| `총걸음` | 200,000 | 오래 배우면 늘 좋을까? (카트폴의 「잊어버림」) |
| `학습률` | 0.0005 | 너무 크면? 너무 작으면? |
| `탐험기간` | 0.3 | 탐험을 일찍 멈추면? |
| `감마` | 0.99 | 착지 보상이 먼 미래라면? |
| `보상가공` | 없음 | 연료를 아끼게? 똑바로 서게? 대신 리워드 해킹 주의! |

In [ ]:
총걸음, 학습률, 탐험기간, 감마, 씨앗 = 200_000, 5e-4, 0.3, 0.99, 0     # ✏️ 바꿔 보세요

def 보상가공(이전상태, 행동, 보상, 다음상태, 끝):
    return 보상                                    # ✏️ 예: 기울기가 크면 조금 빼기 → 보상 - 0.5 * abs(다음상태[4])

내두뇌, _, _ = DQN학습(착륙환경(보상가공), 8, 4, 총걸음=총걸음, 학습률=학습률, 감마=감마, 탐험기간=탐험기간, 목표주기=500, 배치=128,
                      기억크기=100_000, 씨앗=씨앗, 보고=40_000, 평가=착륙시험)
import hashlib, datetime
점수카드 = 착륙시험(내두뇌, range(7000, 7005))                       # 대회 채점용 착륙장 5곳 (원래 보상)
지문 = hashlib.sha256(b''.join(p.detach().numpy().tobytes() for p in 내두뇌.parameters())).hexdigest()[:10]
display(HTML(f'<div style="font-family:sans-serif;border:2px solid #b8ff5c;background:#06080b;color:#fff;padding:18px 22px;max-width:520px">'
             f'<div style="color:#8fd3ff;font-size:12px;letter-spacing:.12em">AI CITY BUILDERS · 강화학습 컵 #4 · 착륙</div>'
             f'<div style="font-size:42px;font-weight:900;margin:6px 0">{점수카드:.1f}점</div>'
             f'<div style="color:#97a3b0;font-size:14px">착륙장 5곳 평균 (원래 보상) · 두뇌 지문 <b style="color:#fff">{지문}</b> · {datetime.date.today()}</div>'
             f'<div style="color:#97a3b0;font-size:13px;margin-top:6px">총걸음 {총걸음:,} · 학습률 {학습률} · 탐험기간 {탐험기간} · 감마 {감마} · 씨앗 {씨앗}</div></div>'))

## CELL 11 · 🤗 허깅페이스 준비 — 토큰 넣기

허깅페이스(huggingface.co)는 전 세계 인공지능 모델이 모이는 곳입니다. 내가 만든 두뇌를 여기 올리면 **누구나 보고, 내려받아 쓸 수 있는 공개 모델**이 됩니다.

1. [huggingface.co](https://huggingface.co) 무료 가입
2. 오른쪽 위 프로필 → **Settings → Access Tokens → Create new token** → 종류 **Write** → 만들기 → 복사
3. 코랩 왼쪽 **🔑 (Secrets)** → `HF_TOKEN` 이름으로 붙여넣고 「노트북 액세스」 켜기
   (또는 아래 칸을 실행하면 나오는 입력창에 붙여넣어도 됩니다)

> ⚠️ 토큰은 비밀번호와 같습니다. 화면에 보이게 붙여넣거나 다른 사람에게 보내지 마세요.

In [ ]:
from huggingface_hub import login, HfApi
토큰 = None
try:
    from google.colab import userdata; 토큰 = userdata.get('HF_TOKEN')
except Exception: pass
if not 토큰:
    토큰 = os.environ.get('HF_TOKEN')
if not 토큰:
    try:
        from getpass import getpass; 토큰 = getpass('허깅페이스 Write 토큰을 붙여넣으세요 (화면에 안 보입니다): ').strip() or None
    except Exception: 토큰 = None
if 토큰:
    login(token=토큰, add_to_git_credential=False); 나 = HfApi().whoami()['name']
    print(f'✅ 로그인 완료 — 내 허깅페이스 이름: {나}')
else:
    나 = None; print('⏭️ 토큰이 없어 업로드는 건너뜁니다. (학습한 두뇌는 그대로 있습니다)')

## CELL 12 · 모델 카드 쓰고 올리기

모델만 올리면 아무도 못 씁니다. **모델 카드(README)** 에 「무엇을 하는 두뇌인지 · 상태와 행동 · 점수 · 학습 설정 · 쓰는 법」을 적어야 합니다. 이력서가 아니라 **작품**으로 나를 알리는 겁니다.

올라가는 파일: `model.pt`(두뇌) · `config.json`(구조와 설정) · `README.md`(모델 카드) · `replay.mp4`(플레이 영상)

In [ ]:
def 허깅페이스에올리기(두뇌, 저장소이름, 게임, 상태설명, 행동설명, 점수이름, 점수, 영상파일, 설정, 한줄소개):
    if not 나: print('⏭️ 로그인하지 않아 건너뜁니다.'); return None
    폴더 = f'hf_{저장소이름}'; os.makedirs(폴더, exist_ok=True)
    torch.save(두뇌.state_dict(), f'{폴더}/model.pt')
    층 = [m for m in 두뇌 if isinstance(m, nn.Linear)]
    구조 = {'type': 'mlp', 'inputs': 층[0].in_features, 'hidden': 층[0].out_features, 'actions': 층[-1].out_features}
    json.dump({'game': 게임, 'architecture': 구조, 'state': 상태설명, 'actions': 행동설명, 'train': 설정, 'score': {점수이름: 점수}},
              open(f'{폴더}/config.json', 'w'), ensure_ascii=False, indent=2)
    if 영상파일 and os.path.exists(영상파일): import shutil; shutil.copy(영상파일, f'{폴더}/replay.mp4')
    카드 = f"""---
library_name: pytorch
tags:
- reinforcement-learning
- deep-reinforcement-learning
- dqn
- custom-implementation
- aicitybuilders
model-index:
- name: {저장소이름}
  results:
  - task:
      type: reinforcement-learning
      name: reinforcement-learning
    dataset:
      name: {게임}
      type: {게임}
    metrics:
    - type: mean_reward
      value: {점수}
      name: {점수이름}
---

# 🧠 {저장소이름}

{한줄소개}

AI CITY BUILDERS 강화학습 AI 5강에서 **DQN(Deep Q-Network)** 으로 직접 학습시킨 두뇌입니다.

| 항목 | 내용 |
|---|---|
| 게임 | {게임} |
| 상태 (입력) | {상태설명} |
| 행동 (출력) | {', '.join(행동설명)} |
| 두뇌 구조 | 입력 {구조['inputs']} → {구조['hidden']} → {구조['hidden']} → 행동 {구조['actions']} (ReLU) |
| 결과 | **{점수이름}: {점수}** |
| 학습 설정 | {', '.join(f'{k} {v}' for k, v in 설정.items())} |

## 플레이 영상
`replay.mp4` 를 열어 보세요.

## 쓰는 법
```python
import torch, torch.nn as nn
from huggingface_hub import hf_hub_download
경로 = hf_hub_download('{나}/{저장소이름}', 'model.pt')
두뇌 = nn.Sequential(nn.Linear({구조['inputs']}, {구조['hidden']}), nn.ReLU(), nn.Linear({구조['hidden']}, {구조['hidden']}), nn.ReLU(), nn.Linear({구조['hidden']}, {구조['actions']}))
두뇌.load_state_dict(torch.load(경로, map_location='cpu'))
행동 = int(두뇌(torch.tensor(상태)).argmax())   # 상태 = 위 표의 입력 숫자들
```

## 배운 곳
[AI CITY BUILDERS 강화학습 AI](https://www.aicitybuilders.com/robot) · Connect AI LAB
"""
    open(f'{폴더}/README.md', 'w').write(카드)
    api = HfApi(); 주소 = f'{나}/{저장소이름}'
    api.create_repo(주소, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path=폴더, repo_id=주소, repo_type='model', commit_message='DQN 두뇌 올리기 (AI CITY BUILDERS 강화학습 5강)')
    print(f'🎉 올라갔습니다 → https://huggingface.co/{주소}'); return 주소

# 루나랜더 착륙 두뇌 올리기 (기본). 카트폴 · 벽돌깨기도 올리려면 아래 주석(#)을 지우세요.
올린주소 = 허깅페이스에올리기(착륙두뇌, 'dqn-lunarlander', 'LunarLander-v3',
    '숫자 8개 — 위치 x · y, 속도 x · y, 기울기, 회전 속도, 왼다리 닿음, 오른다리 닿음', ['쉬기', '왼쪽 엔진', '가운데 엔진', '오른쪽 엔진'],
    '처음 보는 10판 평균 점수 (200 이상 = 성공)', round(float(착륙최종), 1), 'lunar.mp4',
    {'총걸음': 200000, '은닉': 128, '학습률': 0.0005, '감마': 0.99, '탐험기간': 0.3, '목표주기': 500, '배치': 128},
    'Gymnasium 루나랜더를 착륙시키는 두뇌입니다.')
# 허깅페이스에올리기(카트폴두뇌, 'dqn-cartpole', 'CartPole-v1', '숫자 4개 — 수레 위치 · 수레 속도 · 막대 각도 · 막대 각속도', ['왼쪽으로 밀기', '오른쪽으로 밀기'],
#     '처음 보는 10판 평균 점수 (최대 500)', round(float(카트폴최종), 1), 'cartpole.mp4',
#     {'총걸음': 50000, '은닉': 128, '학습률': 0.001, '감마': 0.99, '탐험기간': 0.2, '목표주기': 200, '배치': 128}, '막대가 쓰러지지 않게 수레를 미는 두뇌입니다.')
# 허깅페이스에올리기(벽돌두뇌, 'dqn-mini-breakout', 'AICB-MiniBreakout', '숫자 8개 — 판 위치 · 공 x · 공 y · 공 가로속도 · 공 세로속도 · 공-판 · 남은 벽돌 · 목숨',
#     벽돌행동, '대회 씨앗 5판 평균 벽돌 수 (60개 중)', round(float(벽돌최종), 1), 'breakout.mp4',
#     {'총걸음': 150000, '은닉': 64, '학습률': 0.0005, '감마': 0.99, '탐험기간': 0.4}, '미니 벽돌깨기를 하는 두뇌입니다. aicitybuilders.com/dqn 과 같은 규칙.')
#     {'총걸음': 60000, '보상': '앞으로 간 만큼 +100 · 사고 −5'}, '보상 설계로 리워드 해킹을 피해 트랙을 도는 두뇌입니다.')

## CELL 13 · 내려받아 다시 쓰기 — 진짜 공개 모델인지 확인

방금 올린 두뇌를 **허깅페이스에서 다시 내려받아** 루나랜더를 착륙시켜 봅니다. 다른 사람도 똑같이 내 두뇌를 쓸 수 있다는 뜻입니다.

In [ ]:
if 올린주소:
    from huggingface_hub import hf_hub_download
    경로 = hf_hub_download(올린주소, 'model.pt', force_download=True)
    받은두뇌 = 두뇌만들기(8, 4); 받은두뇌.load_state_dict(torch.load(경로, map_location='cpu'))
    print(f'⬇️ {올린주소} 에서 받은 두뇌 → 처음 보는 5판 평균 {착륙시험(받은두뇌):.0f}점')
else:
    print('⏭️ 업로드를 건너뛰어 이 칸도 건너뜁니다.')

## 정리 — 5강에서 한 것

| | 표(4강까지) | 신경망(DQN, 5강) |
|---|---|---|
| Q 를 얻는 법 | 칸을 찾아 꺼낸다 | 숫자를 넣어 계산한다 |
| 처음 본 상태 | 모른다 (0) | 비슷한 상태에서 배운 걸로 추측 |
| 상태 | 64칸 같은 작은 세상 | 연속된 숫자 · 화면 전체 |

- **DQN = 신경망 + 탐험(2강) + 기억 상자(3강) + γ(4강) + 목표 두뇌(5강)**
- 같은 함수 하나로 카트폴 · 벽돌깨기 · **진짜 루나랜더**를 풀었습니다. 바뀐 건 **환경**뿐.
- Gymnasium 표준 환경이라 다른 사람이 만든 시뮬레이터에도 내 두뇌를 그대로 붙일 수 있습니다.
- 그리고 내 두뇌를 허깅페이스에 올렸습니다 — 이제 여러분은 **공개 모델을 가진 인공지능 개발자**입니다.

> 🔒 이 노트북과 코드는 Connect AI LAB 이 만들었습니다. 공부에는 마음껏 쓰시고, 다른 강의를 만드는 데는 쓰지 말아 주세요.